In [68]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
from itables import init_notebook_mode
from open_dataset_store import quick_start
import plotly.express as px
store = quick_start('./ExperimentResults', backend='local')

init_notebook_mode(all_interactive=True)

# import itables.options as opt
# opt.lengthMenu = [10, 25, 50]
# opt.scrollX = True

CSV_PATH = './results/state_log.csv'

Store initialised at: ./ExperimentResults (Backend: local)


In [69]:
df = pd.read_csv(CSV_PATH)
df = df.copy()
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)
df['timestamp'] = df['Datetime'].astype('int64') // 10**9
ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
print(f'Loaded {len(df)} timesteps, columns: {len(df.columns)}')

list(df.columns)
# df.head(n=20)
# sum = store.get_df_summary(df, detailed=True)


Loaded 768 timesteps, columns: 318


['DayOfYear',
 'Hour',
 'Minute',
 'SPACE1-1_MPC_Time_ms',
 'SPACE1-1_MPC_Status',
 'SPACE1-1_EKF_x_T_in',
 'SPACE1-1_EKF_P_T_in',
 'SPACE1-1_EKF_x_T_m',
 'SPACE1-1_EKF_P_T_m',
 'SPACE1-1_EKF_x_W_in',
 'SPACE1-1_EKF_P_W_in',
 'SPACE1-1_EKF_x_C_in',
 'SPACE1-1_EKF_P_C_in',
 'SPACE1-1_EKF_x_d_T',
 'SPACE1-1_EKF_P_d_T',
 'SPACE1-1_EKF_x_d_W',
 'SPACE1-1_EKF_P_d_W',
 'SPACE1-1_EKF_x_N_occ',
 'SPACE1-1_EKF_P_N_occ',
 'SPACE1-1_EKF_x_alpha_ext',
 'SPACE1-1_EKF_P_alpha_ext',
 'SPACE1-1_EKF_x_alpha_int',
 'SPACE1-1_EKF_P_alpha_int',
 'SPACE1-1_EKF_x_beta_air',
 'SPACE1-1_EKF_P_beta_air',
 'SPACE1-1_EKF_x_beta_mass',
 'SPACE1-1_EKF_P_beta_mass',
 'SPACE1-1_EKF_K_T_in',
 'SPACE1-1_EKF_K_W_in',
 'SPACE1-1_EKF_K_C_in',
 'SPACE1-1_EKF_K_N_occ_from_C_in',
 'SPACE1-1_ideal_temp',
 'SPACE1-1_ideal_hum',
 'SPACE1-1_ideal_co2',
 'SPACE1-1_u_cmd',
 'SPACE1-1_saturation_index',
 'SPACE2-1_MPC_Time_ms',
 'SPACE2-1_MPC_Status',
 'SPACE2-1_EKF_x_T_in',
 'SPACE2-1_EKF_P_T_in',
 'SPACE2-1_EKF_x_T_m',
 'SPACE2-

In [70]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# 1. AHU Monitoring (Updated with Setpoints, Fan Flows, & Power)
# ---------------------------------------------------------
def plot_ahu_monitoring(df):
    """
    Creates subplots to monitor Temperature, Humidity, Air Flow, CO2, and Power 
    across the Air Handling Unit, including setpoints.
    """
    stages = ['Outdoor_Air', 'Relief_Air', 'Mixer_Inlet', 'Mixed_Air', 'CC_Out', 'HC_Out', 'Fan_Out']
    
    # Added a 5th row for Power
    fig = make_subplots(
        rows=5, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.04,
        subplot_titles=('Temperatures & Setpoints (°C)', 'Relative Humidity (%)', 'Air Flow & Setpoints (kg/s)', 'CO2 Levels (ppm)', 'AHU Power Consumption (W)')
    )

    # 1. Temperature (Including CC and HC Setpoints)
    for stage in stages:
        col = f'{stage}_Temp_C'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=1, col=1)
            
    # Add Temperature Setpoints as dashed lines
    for col in ['Act_CC_Temp_SP_C', 'Act_HC_Temp_SP_C']:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines', line=dict(dash='dash')), row=1, col=1)

    # 2. Humidity
    for stage in stages:
        col = f'{stage}_RH_pct'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=2, col=1)

    # 3. Flow (Including Fan Flow and OA Setpoints)
    for stage in stages:
        col = f'{stage}_Flow_kg_s'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=3, col=1)
            
    # Add Air Flow Setpoints as dashed lines
    for col in ['Act_Fan_Flow_kg_s', 'Act_OA_Flow_SP_kg_s']:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines', line=dict(dash='dash')), row=3, col=1)

    # 4. CO2
    for stage in stages:
        col = f'{stage}_CO2_ppm'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=4, col=1)
            
    # 5. Power (CC, HC, Fan Power)
    for col in ['CC_Power_W', 'HC_Power_W', 'Fan_Power_W']:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=5, col=1)

    fig.update_layout(height=1200, title_text="AHU System Monitoring (Incl. Setpoints & Power)", hovermode="x unified")
    fig.show()


# ---------------------------------------------------------
# 2. Zone Monitoring (Updated with Reheat & Flow Setpoints)
# ---------------------------------------------------------
def plot_zone_monitoring(df, zone_name="SPACE5-1"):
    """
    Plots all relevant metrics for a specifically chosen zone.
    Example zone_names: 'SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1'
    """
    # Extract columns that belong to the selected zone
    zone_cols = [col for col in df.columns if zone_name in col]
    
    # Group them by metric type (Now capturing Reheat Setpoints and Flow Setpoints)
    temp_cols = [c for c in zone_cols if 'Temp_C' in c or 'T_m_C' in c or 'Reheat_SP_C' in c]
    rh_cols = [c for c in zone_cols if 'RH_pct' in c]
    flow_cols = [c for c in zone_cols if 'Flow' in c]
    co2_cols = [c for c in zone_cols if 'CO2' in c]
    power_cols = [c for c in zone_cols if 'Load_W' in c or 'Reheater_W' in c]
    occupant_cols = [c for c in zone_cols if 'Occupants' in c]
    
    fig = make_subplots(
        rows=5, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.04,
        subplot_titles=(
            f'{zone_name} Temperatures & Reheat SP', 
            f'{zone_name} Humidity & CO2', 
            f'{zone_name} Air Flow & Flow SP', 
            f'{zone_name} Power (Reheat & Equip)',
            f'{zone_name} Occupancy'
        )
    )

    # 1. Temp (Dash the setpoint line)
    for col in temp_cols:
        line_style = dict(dash='dash') if 'SP' in col else dict(dash='solid')
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, line=line_style), row=1, col=1)
        
    # 2. RH & CO2
    for col in rh_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=2, col=1)
    for col in co2_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, line=dict(dash='dot')), row=2, col=1)
        
    # 3. Flow (Dash the setpoint line)
    for col in flow_cols:
        line_style = dict(dash='dash') if 'SP' in col else dict(dash='solid')
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, line=line_style), row=3, col=1)
        
    # 4. Power
    for col in power_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=4, col=1)
        
    # 5. Occupants
    for col in occupant_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, fill='tozeroy'), row=5, col=1)

    fig.update_layout(height=1200, title_text=f"Comprehensive Log for {zone_name}", hovermode="x unified")
    fig.show()

# ---------------------------------------------------------
# 3. Energy Consumption (Unchanged - ready for use)
# ---------------------------------------------------------
def plot_energy_consumption(df):
    energy_cols = ['Meter_Bldg_Elec_J', 'Meter_HVAC_Elec_J', 'Meter_AHU_Elec_J', 'Meter_Bldg_Gas_J']
    
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.1,
        subplot_titles=('Interval Energy Consumption (Joules)', 'Cumulative Energy Consumption (Joules)')
    )

    for col in energy_cols:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=1, col=1)

    for col in energy_cols:
        if col in df.columns:
            cumulative_series = df[col].cumsum()
            fig.add_trace(go.Scatter(x=df['timestamp'], y=cumulative_series, name=f'{col} (Cumulative)', mode='lines'), row=2, col=1)

    fig.update_layout(height=700, title_text="Building Energy Meters", hovermode="x unified")
    fig.show()


def plot_ekf_performance(df, zone_name="SPACE1-1"):
    """
    Plots the EKF state estimates against the true measured values for a given zone
    to monitor filter tracking performance and convergence.
    """
    
    # Define the mapping between EKF estimated states and the True measurements
    state_mappings = [
        {
            'title': f'{zone_name} Indoor Air Temp (°C)',
            'ekf_col': f'{zone_name}_EKF_x_T_in',
            'true_col': f'{zone_name}_Temp_C'
        },
        {
            'title': f'{zone_name} Thermal Mass Temp (°C)',
            'ekf_col': f'{zone_name}_EKF_x_T_m',
            'true_col': f'{zone_name}_T_m_C'
        },
        {
            'title': f'{zone_name} Humidity Ratio (kg/kg)',
            'ekf_col': f'{zone_name}_EKF_x_W_in',
            'true_col': f'{zone_name}_W_in_kg_kg'
        },
        {
            'title': f'{zone_name} CO2 Concentration (ppm)',
            'ekf_col': f'{zone_name}_EKF_x_C_in',
            'true_col': f'{zone_name}_CO2_ppm'
        },
        {
            'title': f'{zone_name} Occupancy Count',
            'ekf_col': f'{zone_name}_EKF_x_N_occ',
            'true_col': f'{zone_name}_Occupants'
        }
    ]
    
    # Create a 5-row subplot
    fig = make_subplots(
        rows=5, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.04,
        subplot_titles=[mapping['title'] for mapping in state_mappings]
    )

    # Loop through each state mapping and add the traces to the respective subplot row
    for i, mapping in enumerate(state_mappings, start=1):
        ekf_col = mapping['ekf_col']
        true_col = mapping['true_col']
        
        # Plot True Value (Solid Line)
        if true_col in df.columns:
            fig.add_trace(
                go.Scatter(x=df['timestamp'], y=df[true_col], name=f'True {true_col}', mode='lines', line=dict(color='blue')), 
                row=i, col=1
            )
            
        # Plot EKF Estimate (Dashed Line)
        if ekf_col in df.columns:
            fig.add_trace(
                go.Scatter(x=df['timestamp'], y=df[ekf_col], name=f'EKF {ekf_col}', mode='lines', line=dict(color='orange', dash='dash')), 
                row=i, col=1
            )

    fig.update_layout(
        height=1200, 
        title_text=f"EKF State Estimation vs Ground Truth Validation for {zone_name}", 
        hovermode="x unified"
    )
    
    fig.show()

def plot_ekf_internals(df, zone_name="SPACE1-1"):
    """
    Plots the EKF internal variables: Estimated disturbances, adaptive parameters, 
    and error covariances. Utilizes a secondary Y-axis for parameters to handle 
    scale differences between Alphas and Betas.
    """
    # Define subplot structure and explicitly enable secondary_y for Row 2
    fig = make_subplots(
        rows=3, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.06,
        subplot_titles=(
            f'{zone_name} Estimated Disturbances (Temp & Humidity)', 
            f'{zone_name} Adaptive Parameters (Alphas & Betas)',
            f'{zone_name} Error Covariance (Confidence)'
        ),
        specs=[
            [{"secondary_y": False}], # Row 1: Disturbances
            [{"secondary_y": True}],  # Row 2: Parameters (needs dual axis)
            [{"secondary_y": False}]  # Row 3: Covariances
        ]
    )
    
    # 1. Disturbances
    for col in [f'{zone_name}_EKF_x_d_T', f'{zone_name}_EKF_x_d_W']:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=1, col=1)

    # 2. Adaptive Parameters (Alphas and Betas)
    param_cols = [
        f'{zone_name}_EKF_x_alpha_ext', 
        f'{zone_name}_EKF_x_alpha_int',
        f'{zone_name}_EKF_x_beta_air', 
        f'{zone_name}_EKF_x_beta_mass'
    ]
    
    for col in param_cols:
        if col in df.columns:
            is_beta = 'beta' in col
            line_style = dict(dash='dash') if is_beta else dict(dash='solid')
            
            # Map betas to the secondary y-axis, alphas to the primary
            fig.add_trace(
                go.Scatter(x=df['timestamp'], y=df[col], name=col, line=line_style), 
                row=2, col=1, 
                secondary_y=is_beta 
            )
            
    # 3. Covariances (P values)
    p_cols = [c for c in df.columns if f'{zone_name}_EKF_P_' in c]
    for col in p_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=3, col=1)
        
    # Optional: Label the dual Y-axes on Row 2 to make it clear
    fig.update_yaxes(title_text="Alphas", row=2, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Betas", row=2, col=1, secondary_y=True)
        
    # Keep the taller height to accommodate the three rows
    fig.update_layout(height=900, title_text=f"EKF Internal Mechanics for {zone_name}", hovermode="x unified")
    fig.show()


In [71]:
plot_ahu_monitoring(df)

In [72]:
plot_energy_consumption(df)

In [89]:
zone_name="SPACE5-1" 

In [90]:
plot_zone_monitoring(df, zone_name=zone_name)

In [91]:
# To see how well the filter is tracking the real simulation states
plot_ekf_performance(df, zone_name=zone_name)

In [92]:
# To look at the unmeasured disturbances the filter is calculating, or to see if covariance (P) is converging
plot_ekf_internals(df, zone_name=zone_name)